# AI Powered Network Intrusion Detection System (AI-NIDS)
## Anomaly Detection Model Training (Intel Arc / CPU Accelerated)

This notebook trains a **Deep Autoencoder Anomaly Detection Model** on the processed **CICIDS2017** dataset.
- **Training approach**: Trained strictly on **BENIGN (Normal)** traffic. The model learns to reconstruct normal traffic with low error.
- **Anomaly Detection**: Anomalous/Attack traffic results in high reconstruction error (MSE), allowing precise intrusion detection.
- **Hardware Acceleration**: Automatically utilizes **Intel Arc Graphics (`torch.xpu`)** if available, or falls back to **CPU**.

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, f1_score

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Hardware Device Setup (Intel Arc GPU 'xpu' or CPU)
def get_device():
    if hasattr(torch, 'xpu') and torch.xpu.is_available():
        device = torch.device('xpu')
        print(f"Using Hardware Acceleration: Intel Arc GPU (XPU: {torch.xpu.get_device_name(0)})")
    elif torch.cuda.is_available():
        device = torch.device('cuda')
        print(f"Using Hardware Acceleration: NVIDIA GPU ({torch.cuda.get_device_name(0)})")
    else:
        device = torch.device('cpu')
        print("Using Hardware: CPU (Intel Arc XPU not active/available)")
    return device

device = get_device()

In [ ]:
# Load Processed Datasets
data_dir = 'data_processed'

print('Loading preprocessed datasets from', data_dir, '...')
X_train_df = pd.read_parquet(os.path.join(data_dir, 'train_benign.parquet'))
X_val_df = pd.read_parquet(os.path.join(data_dir, 'val_X.parquet'))
y_val = pd.read_parquet(os.path.join(data_dir, 'val_y.parquet'))['is_anomaly'].values
X_test_df = pd.read_parquet(os.path.join(data_dir, 'test_X.parquet'))
y_test = pd.read_parquet(os.path.join(data_dir, 'test_y.parquet'))['is_anomaly'].values

with open(os.path.join(data_dir, 'feature_names.json'), 'r') as f:
    feature_info = json.load(f)

print(f"Train Set (Benign only): {X_train_df.shape}")
print(f"Validation Set: {X_val_df.shape} (Attacks: {y_val.sum():,})")
print(f"Test Set: {X_test_df.shape} (Attacks: {y_test.sum():,})")
print(f"Features count: {feature_info['num_features']}")

In [ ]:
# Convert to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_df.values, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_df.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_df.values, dtype=torch.float32)

batch_size = 512
train_loader = DataLoader(TensorDataset(X_train_tensor, X_train_tensor), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, X_val_tensor), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, X_test_tensor), batch_size=batch_size, shuffle=False)

print('DataLoaders initialized with batch_size =', batch_size)

In [ ]:
# Deep Autoencoder Neural Network for Anomaly Detection
class NIDSAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super(NIDSAutoencoder, self).__init__()
        
        # Encoder: compress high-dimensional flow features into bottleneck representation
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 48),
            nn.BatchNorm1d(48),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.1),
            
            nn.Linear(48, 24),
            nn.BatchNorm1d(24),
            nn.LeakyReLU(0.2),
            
            nn.Linear(24, 12),
            nn.LeakyReLU(0.2)
        )
        
        # Decoder: reconstruct original features from bottleneck
        self.decoder = nn.Sequential(
            nn.Linear(12, 24),
            nn.BatchNorm1d(24),
            nn.LeakyReLU(0.2),
            
            nn.Linear(24, 48),
            nn.BatchNorm1d(48),
            nn.LeakyReLU(0.2),
            
            nn.Linear(48, input_dim)
        )
        
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

input_dim = feature_info['num_features']
model = NIDSAutoencoder(input_dim).to(device)
print(model)

In [ ]:
# Training Autoencoder on Benign Traffic
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

epochs = 20
train_losses = []

print(f"Starting training for {epochs} epochs on {device}...")
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    for batch_x, _ in train_loader:
        batch_x = batch_x.to(device)
        
        optimizer.zero_grad()
        output = model(batch_x)
        loss = criterion(output, batch_x)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch_x.size(0)
        
    avg_loss = total_loss / len(train_loader.dataset)
    train_losses.append(avg_loss)
    scheduler.step(avg_loss)
    
    if epoch % 2 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{epochs:02d}] - Loss (MSE): {avg_loss:.6f}")

torch.save(model.state_dict(), 'autoencoder_nids.pth')
print("Model saved to 'autoencoder_nids.pth'")

In [ ]:
# Compute Reconstruction Loss (MSE per sample) for Validation Set
model.eval()
def compute_reconstruction_error(loader, model, device):
    errors = []
    with torch.no_grad():
        for batch_x, _ in loader:
            batch_x = batch_x.to(device)
            recon = model(batch_x)
            mse = torch.mean((batch_x - recon) ** 2, dim=1)
            errors.extend(mse.cpu().numpy())
    return np.array(errors)

val_errors = compute_reconstruction_error(val_loader, model, device)
roc_auc = roc_auc_score(y_val, val_errors)
print(f"Validation ROC-AUC Score: {roc_auc:.4f}")

# Find Optimal Anomaly Threshold using F1-Score on Validation set
precisions, recalls, thresholds = precision_recall_curve(y_val, val_errors)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else thresholds[-1]
print(f"Optimal Anomaly Reconstruction Threshold: {best_threshold:.6f} (Best Val F1-Score: {f1_scores[best_idx]:.4f})")

# Evaluate on Test Set
test_errors = compute_reconstruction_error(test_loader, model, device)
test_preds = (test_errors > best_threshold).astype(int)

print("\n" + "="*50)
print("         TEST SET EVALUATION REPORT")
print("="*50)
print(classification_report(y_test, test_preds, target_names=['BENIGN (Normal)', 'ATTACK (Anomaly)']))